In [1]:
import sys
import torch

import numpy as np
import trimesh
import plotly.graph_objects as go

In [2]:
sys.path.append("..")

In [3]:
from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

## Info

In [4]:
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

## Hand Models

In [5]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [6]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Source Gripper Pose

In [7]:
grasp_pose = torch.zeros(9)
grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3] = 1
grasp_pose[7] = 1 
print("Pose:", grasp_pose)


grasp_dofs_lower = source_model.dynamic_joints_q_lower.squeeze(0).clone()
grasp_dofs_mid = torch.tensor(source_model.dynamic_joints_q_mid)

grasp_dofs = grasp_dofs_lower + grasp_dofs_mid
# grasp_dofs = grasp_dofs_mid # for shadowhand
# scale = 0.5
# grasp_dofs = scale * torch.rand_like(grasp_dofs_mid) * grasp_dofs_mid + grasp_dofs_lower



print("Dofs:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([0.1000, 0.2000, 0.3000, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000])
Dofs: tensor([-0.3491,  0.5236,  0.8727,  0.8727, -0.6109,  0.5236,  0.8727,  0.8727,
        -0.8727,  0.5236,  0.8727,  0.8727, -0.6109,  0.5236,  0.8727,  0.8727,
         1.0472, -0.6981,  0.8727,  0.8727])


In [8]:
sample_grasp_q.shape

torch.Size([1, 29])

## Grasp Transfer Optimization

In [9]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [10]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [11]:
print(q_traj.shape)
best_q = q_traj[21, -1]
print(best_q.shape)

torch.Size([32, 301, 9])
torch.Size([9])


In [12]:
best_q.shape[0]

9

In [13]:
target_model.dynamic_joints_q_lower.shape

torch.Size([1, 2])

In [14]:
midjoints = torch.tensor(target_model.dynamic_joints_q_mid)
print(midjoints)

tensor([0.0200, 0.0200])


In [15]:
target_model.dynamic_joints_q_upper[0]

tensor([0.0400, 0.0400])

In [16]:
target_model.dynamic_joints_q_lower[0]

tensor([0., 0.])

In [17]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, midjoints), dim=0)

In [18]:
print(best_q)

tensor([ 0.1412,  0.3206,  0.3203, -0.1293, -0.6488, -0.0715, -0.7098,  0.0893,
         0.1789,  0.0200,  0.0200])


## Visualize results

In [19]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
vis_data += target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs/viz_gtransfer_test.html")


Plotting TARGET and SOURCE together...


In [20]:
base_pose = torch.zeros(9)
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
base_pose[3] = 1
base_pose[7] = 1 
print("Base Pose:", base_pose)

base_q = torch.cat((base_pose, midjoints), dim=0)
print("Base Q:", base_q)

Base Pose: tensor([0., 0., 0., 1., 0., 0., 0., 1., 0.])
Base Q: tensor([0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
        0.0200, 0.0200])
